# Nasdaq ITCH Feed Handler v1

In [ ]:
# imports

from pynq import Overlay, allocate, MMIO
import numpy as np
import time
import gzip
import struct
import math

In [ ]:
# Constants

DMA_DATA    = 0x4040
GPIO0       = 0x4120
GPIO1       = 0x4121
GPIO2       = 0x4122    
PATH        = "/home/xilinx/jupyter_notebooks/Nasdaq/name.bit"
DATA_PATH   = "/home/xilinx/jupyter_notebooks/Nasdaq/12302019.NASDAQ_ITCH50.gz"

START_BYTES = 2


In [ ]:
# Overlay Load

ol = Overlay(PATH)
ol.download()
print("Overlay Loaded:")

print(ol.ip_dict.items())


writer = MMIO(ol.ip_dict(["data_handler"]["phys_addr"]),ol.ip_dict(["data_handler"]["addr_range"]))

In [ ]:
bbo_ip = MMIO(0x4000_0000, 0x1000) 

def read_bbo():
    # Read the 4 consecutive 32-bit registers
    bid_price  = bbo_ip.read(0x00) # Offset 0
    bid_shares = bbo_ip.read(0x04) # Offset 4
    ask_price  = bbo_ip.read(0x08) # Offset 8
    ask_shares = bbo_ip.read(0x0C) # Offset 12
    
    return {
        "bid_price": bid_price,
        "bid_shares": bid_shares,
        "ask_price": ask_price,
        "ask_shares": ask_shares
    }

In [ ]:
# Buffer Formation
buffer_0 = allocate(shape=10000, dtype=np.uint8)
buffer_1 = allocate(shape=10000, dtype=np.uint8)
buffer_choice = 0
buffer_count = 0

TARGET_SYMBOL   =   b"APPL      "
target_locate   =   None
print(b"Tracking stock: " + TARGET_SYMBOL)
# Packet data

with gzip.open(DATA_PATH, "rb") as data:
    while True:
        start_msg = data.read(START_BYTES)

        if not start_msg:
            writer.write(DMA_DATA, buffer_1[:buffer_count]) if buffer_choice else writer.write(DMA_DATA, buffer_0[:buffer_count])
            print("All Data Parsed")
            break

        # Find length of instruction
        ins_len =   int.from_bytes(start_msg, byteorder='big')
        output_data = data.read(ins_len)

        if len(output_data) < ins_len:
            break
        msg_type = output_data[0:1]

        if msg_type == b'R':
            actual_symbol = output_data[11:19] 
            if actual_symbol == TARGET_SYMBOL:
                target_locate = output_data[1:3]
                print(f"Found Target! Locate Code Bytes: {target_locate.hex()}")

        keep_message = False
        modified_msg = bytearray(output_data)

        if msg_type == b'S': 
            keep_message = True
        elif target_locate is not None and output_data[1:3] == target_locate:
            # 2. Overwrite Stock Locate to 1
            modified_msg[1:3] = (1).to_bytes(2, byteorder='big')
            
            # 3. Locate the Price Offset
            price_offset = None
            if msg_type in [b'A', b'F', b'C']:
                price_offset = 32
            elif msg_type == b'U':
                price_offset = 31
                
            # 4. Extract, Divide by 100, and Repack the Price
            if price_offset is not None:
                # Extract the 4-byte integer
                original_price = int.from_bytes(modified_msg[price_offset:price_offset+4], byteorder='big')
                
                # Divide by 100 using integer division
                new_price = original_price // 100
                
                # Repack into the bytearray
                modified_msg[price_offset:price_offset+4] = new_price.to_bytes(4, byteorder='big')
                
            keep_message = True

        # Send to DMA using the padded single-transfer approach
        if keep_message:
            pad_len = math.ceil(ins_len / 8) * 8
            padded_msg = bytearray(pad_len)
            padded_msg[:ins_len] = modified_msg
            writer.write(DMA_DATA, padded_msg)

        # Buffer check (if full)
        if(buffer_count + ins_len >= 10000):
            writer.write(DMA_DATA, buffer_1[:buffer_count]) if buffer_choice else writer.write(DMA_DATA, buffer_0[:buffer_count])
            buffer_count = 0
            buffer_choice = 1 - buffer_choice

        # Double buffer selection
        if(buffer_choice):
            tmp_data = np.frombuffer(output_data, dtype=np.uint8)
            buffer_1[buffer_count : buffer_count + ins_len] = tmp_data
            buffer_count += ins_len
        else:
            tmp_data = np.frombuffer(output_data, dtype=np.uint8)
            buffer_0[buffer_count : buffer_count + ins_len] = tmp_data
            buffer_count += ins_len

        current_bbo = read_bbo()
        print(f"Bid: {current_bbo['bid_shares']} @ {current_bbo['bid_price']}")
        print(f"Bid: {current_bbo['ask_shares']} @ {current_bbo['ask_price']}")

            
        


        
